---
title: "Phylogenetic OU Regression — Julia/Turing Tutorial"
format: html
jupyter: julia-1.12
---

# Running a Phylogenetic OU Regression with Julia and Turing.jl

Phylogenetic comparative methods for binary typological features have long relied on two approaches: the continuous-time Markov chain (CTMC) model of Pagel & Meade (2006), which treats features as evolving between discrete states, and Brownian motion random effects, which model continuous phylogenetic drift. Both have shortcomings when applied to typological data. CTMC discards gradient information by forcing evolution into a discrete state space. Brownian motion assumes unbounded drift, yet typological features cluster around cross-linguistic norms rather than diverging without limit — a pattern consistent with stabilizing selection toward typological attractors.

The Ornstein-Uhlenbeck (OU) process directly models this stabilization. It is a Brownian motion with a restoring force toward an attractor, parameterized by a mean-reversion rate $\lambda$. As $\lambda \to 0$ it reduces to Brownian motion; for $\lambda > 0$ it captures the tendency of typological features to remain near cross-linguistic norms regardless of genealogical distance.

This notebook is a Julia/Turing.jl translation of the main R+Stan tutorial. It covers the same sequence of models — from simple baselines through the standard CTMC approach to Brownian and OU phylogenetic models — and compares them via Bayes factors and leave-one-out cross-validation. The OU models consistently outperform the alternatives, both statistically and conceptually.

For multi-tree marginalization (averaging over a posterior sample of 902 EDGE trees), see the companion notebook `phylogenetic_OU_regression_julia.qmd`.

## Preparing the environment

The git repository https://github.com/gerhardJaeger/OU_logistic_example holds the source code for this notebook. It contains a file `OU_logistic_example.yml`. You can use it to set up a conda environment containing all the required packages.

In [ ]:
cd(@__DIR__)
using Pkg
Pkg.activate(".")
Pkg.instantiate()

## Loading the required packages

In [ ]:
using CodecZlib: GzipDecompressor
using CSV
using DataFrames
using DataFramesMeta
using Distributions
using Downloads
using DynamicPPL
using ExponentialUtilities
using LinearAlgebra
using LogDensityProblems
using LogExpFunctions
using MCMCChains
using Optim
using Phylo
using Pipe
using Printf
using Random
using RCall
using Statistics
using StatsBase
using StatsFuns
using StatsPlots
using Main.Threads
using Turing
gr()
include("bridge_sampling.jl")

## Data

We use the EDGE tree (Bouckaert et al. 2022).

In [ ]:
isdir("../data") || mkpath("../data")

tree_gz  = "../data/global-language-tree-MCC-labelled.tree.gz"
tree_file = "../data/global-language-tree-MCC-labelled.tree"

if !isfile(tree_file)
    Downloads.download(
        "https://github.com/rbouckaert/global-language-tree-pipeline/releases/download/v1.0.0/global-language-tree-MCC-labelled.tree.gz",
        tree_gz
    )
    open(tree_gz) do f_in
        open(tree_file, "w") do f_out
            write(f_out, transcode(GzipDecompressor, read(f_in)))
        end
    end
    rm(tree_gz)
end

# The MCC tree is a BEAST-annotated Nexus file; use ape via RCall to read it.
# ape::read.nexus handles the BEAST annotations that parsenewick cannot parse.
R"""
library(ape)
.full_tree <- read.nexus($tree_file)
.full_tree$tip.label <- sapply(strsplit(.full_tree$tip.label, "_"), `[`, 1)
.full_tree$edge.length <- .full_tree$edge.length / mean(.full_tree$edge.length)
"""

# All tip labels of the full tree (needed for Grambank filtering below)
all_tip_labels = rcopy(R"as.character(.full_tree$tip.label)")
println("Full tree loaded: ", length(all_tip_labels), " tips")

It is advisable to rescale the tree so that the mean branch length is 1. This is already done in R above.

## Grambank data

We look at the following feature pair, a classic test case in word-order typology:

- **GB193**: Order of adnominal property word and noun (1 = NAdj, 2 = AdjN)
- **GB133**: Is the pragmatically unmarked order verb-final for transitive clauses? (0 = no, 1 = yes)

First, we load and filter the Grambank data.

In [ ]:
grambank_file = "../data/grambank_vals.csv"
if !isfile(grambank_file)
    Downloads.download(
        "https://raw.githubusercontent.com/grambank/grambank/refs/heads/master/cldf/values.csv",
        grambank_file
    )
end

vals = CSV.read(grambank_file, DataFrame)

Now we filter to the two parameters of interest and the languages in the tree.

In [ ]:
tip_names = Set(all_tip_labels)

d = @pipe vals |>
    filter(r -> r.Language_ID ∈ tip_names, _) |>
    filter(r -> r.Parameter_ID ∈ ["GB193", "GB133"], _) |>
    select(_, [:Language_ID, :Parameter_ID, :Value]) |>
    dropmissing |>
    filter(r -> r.Value ∈ ["0", "1", "2"], _) |>
    transform(_, :Value => (v -> parse.(Int, v)) => :Value) |>
    unstack(_, :Parameter_ID, :Value) |>
    dropmissing |>
    filter(r -> r.GB193 > 0, _)

# Convert to 0/1
d[!, :x] = d.GB193 .- 1
d[!, :y] = d.GB133

For now, we keep the data set small and restrict ourselves to 100 taxa. We then prune the
tree in R using `ape::drop.tip`, export as a Newick string, and re-read with `parsenewick`.
We also extract the tip labels, edge matrix, edge lengths, and patristic distance matrix
from the pruned ape tree object.

In [ ]:
if isfile("../data/grambank_vals_pruned.csv")
    d = CSV.read("../data/grambank_vals_pruned.csv", DataFrame)
else
    Random.seed!(123)
    sample_idx = sort(sample(1:nrow(d), 100; replace=false))
    d = d[sample_idx, :]
    CSV.write("../data/grambank_vals_pruned.csv", d)
end

# Prune tree in R to only the languages in d, then roundtrip via Newick
keep_langs = d.Language_ID
R"""
.pruned <- drop.tip(.full_tree, setdiff(.full_tree$tip.label, $keep_langs))
.pruned <- reorder(.pruned, "postorder")
.tip_labels  <- .pruned$tip.label
.edge_matrix <- .pruned$edge
.edge_lengths <- .pruned$edge.length
.dist_matrix <- cophenetic.phylo(.pruned)
.newick_str  <- write.tree(.pruned)
"""

# Pull extracted tree data into Julia
tip_labels_r  = rcopy(R"as.character(.tip_labels)")
edge_matrix_r = rcopy(R"matrix(as.integer(.edge_matrix), ncol=2)")
edge_lengths_r = rcopy(R"as.numeric(.edge_lengths)")
dist_matrix_r  = rcopy(R"as.matrix(.dist_matrix)")
newick_str     = rcopy(R".newick_str")

# Build a Phylo.jl RootedTree from the Newick string so downstream code is unchanged
edge_tree = parsenewick(newick_str)

# Reorder d to match the tip order in the pruned tree
taxa_ordered = tip_labels_r
idx = indexin(taxa_ordered, d.Language_ID)
d = d[idx, :]
taxa = taxa_ordered

println("Number of languages: ", nrow(d))
println("Prevalence of x=1: ", mean(d.x))
println("Prevalence of y=1: ", mean(d.y))
println("Number of tree tips: ", nleaves(edge_tree))

## Tree decomposition helpers

All phylogenetic models share the same tree encoding. We decompose the tree into arrays of mothers, daughters, and branch lengths in postorder, which allows us to propagate states from root to tips in a single forward pass.

In [ ]:
function decompose_tree(tree::RootedTree)
    root = getroot(tree).name
    mothers = String[]
    daughters = String[]
    lengths = Float64[]
    for nd in traversal(tree, postorder)
        if !isroot(tree, nd)
            mother = getparent(tree, nd)
            br = getinbound(tree, nd)
            bl = getlength(tree, br)
            push!(mothers, mother.name)
            push!(daughters, nd.name)
            push!(lengths, bl)
        end
    end
    nodes = sort!(unique(vcat(mothers, daughters)))
    nodes_dict = Dict{String,Int}(nodes .=> 1:length(nodes))
    (root=root, mothers=mothers, daughters=daughters, lengths=lengths, nodes_dict=nodes_dict)
end

tree_info = decompose_tree(edge_tree)
println("Number of nodes: ", length(tree_info.nodes_dict))
println("Number of edges: ", length(tree_info.lengths))

We also precompute the Cholesky factor of the phylogenetic covariance matrix (VCV) for use in the Brownian motion model.

In [ ]:
function get_vcv(tree; taxa=taxa)
    root = getroot(tree)
    tips = getleaves(tree)
    tip_names = getleafnames(tree)
    n = length(tips)
    vcv = zeros(n, n)
    for i in 1:n
        for j in i:n
            if i == j
                vcv[i, j] = distance(tree, root, tips[i])
            else
                vcv[j, i] = vcv[i, j] = distance(tree, root, mrca(tree, [tips[i], tips[j]]))
            end
        end
    end
    return vcv[indexin(tip_names, taxa), indexin(tip_names, taxa)]
end

scaled_vcv = get_vcv(edge_tree) ./ 100
scaled_chol = cholesky(scaled_vcv).L

## Baseline models (no phylogenetic control)

We begin with models that ignore phylogenetic structure entirely. These serve as baselines and confirm that the data contain a signal before asking whether phylogenetic control is warranted.

### Vanilla logistic regression

The simplest model is a Bernoulli logistic regression with a centered predictor.

$$
\begin{align}
\alpha &\sim \text{Student-t}(3, 0, 2.5)\\
\beta &\sim \mathcal N(0, 2)\\
\bar{x} &:= \frac{1}{N}\sum_{i}x_i\\
y_i &\sim \text{Bernoulli}(\text{logit}^{-1}(\alpha + \beta (x_i - \bar{x})))
\end{align}
$$

In [ ]:
x_centered = d.x .- mean(d.x)

@model function vanilla_regression(x_centered, y)
    N = length(y)
    alpha ~ TDist(3) * 2.5
    beta ~ Normal(0, 2)
    for i in 1:N
        y[i] ~ BernoulliLogit(alpha + beta * x_centered[i])
    end
end

In [ ]:
Random.seed!(42)
chain_vanilla_regression = Turing.sample(
    vanilla_regression(x_centered, d.y),
    NUTS(0.65),
    MCMCThreads(),
    1_000,
    4
)
describe(chain_vanilla_regression)

In [ ]:
plot(chain_vanilla_regression, size=(700, 400))

The posterior of $\beta$ indicates the direction of the relationship between adjective-noun order and verb-final order.

#### Bridge sampling for marginal likelihood

We use bridge sampling to estimate the log marginal likelihood. This requires extracting the unconstrained parameter samples and passing them to the log-density function.

In [ ]:
model_vanilla_regression = vanilla_regression(x_centered, d.y)
ldf_vanilla_regression = LogDensityFunction(model_vanilla_regression)
param_names_vr = names(chain_vanilla_regression, :parameters)
samples_vr = Array(chain_vanilla_regression[:, param_names_vr, :])
f_vr(x) = LogDensityProblems.logdensity(ldf_vanilla_regression, x)

logml_vanilla_regression = bridge_sampling(samples_vr, f_vr)
println("Log marginal likelihood (vanilla regression): ", logml_vanilla_regression)

#### LOO log-likelihoods

In [ ]:
loglik_vanilla_regression = hcat([
    logpdf.(BernoulliLogit.(
        Array(chain_vanilla_regression[:, :alpha, :]) .+
        Array(chain_vanilla_regression[:, :beta, :]) .* x_centered[i]
    ), d.y[i])
    for i in 1:nrow(d)
]...)

loo_vanilla_regression = convert(
    Dict,
    R"library(loo); loo($(loglik_vanilla_regression))"
)
println("LOOIC (vanilla regression): ", loo_vanilla_regression["looic"])

### Vanilla logistic correlation

A regression model assumes an asymmetry between predictor and response that is hard to justify for observational data. A symmetric model treats both variables as generated by a joint stochastic process:

$$
\begin{align}
z_i &\sim \mathcal N(\mu, \Sigma)\\
\Sigma &= \text{diag}(\sigma) \cdot \begin{bmatrix}1 & \rho\\ \rho & 1\end{bmatrix} \cdot \text{diag}(\sigma)\\
\mu_k &\sim \mathcal N(0, 2),\quad \sigma_k \sim \text{Lognormal}(0,1),\quad \rho_u \sim \mathcal N(0,1)\\
x_i &\sim \text{Bernoulli}(\text{logit}^{-1}(z_{i,1})),\quad
y_i &\sim \text{Bernoulli}(\text{logit}^{-1}(z_{i,2}))
\end{align}
$$

Here $\rho$ is the latent correlation between the two features. We transform an unconstrained normal variate $\rho_u$ to $(-1,1)$ via the normal CDF: $\rho = 2\Phi(\rho_u)-1$.

In [ ]:
@model function vanilla_correlation(N)
    mu ~ MvNormal(zeros(2), 2.0)
    rho_u ~ Normal()
    rho := 2 * cdf(Normal(), rho_u) - 1
    sigma_u ~ filldist(Normal(0.0, 1.0), 2)
    sigma := exp.(sigma_u)

    Sigma_l = [
        1.0  0.0
        rho  sqrt(1 - rho^2)
    ]

    z_std ~ filldist(MvNormal(zeros(2), 1.0), N)
    z := diagm(sigma) * Sigma_l * z_std .+ mu

    x ~ product_distribution(BernoulliLogit.(z[1, :]))
    y ~ product_distribution(BernoulliLogit.(z[2, :]))
end

In [ ]:
Random.seed!(42)
chain_vanilla_correlation = Turing.sample(
    vanilla_correlation(nrow(d)) | (x=d.x, y=d.y),
    NUTS(0.65),
    MCMCThreads(),
    1_000,
    4
)

nms_vc = [nm for nm in names(chain_vanilla_correlation)
          if !occursin("z", string(nm)) && !occursin("_u", string(nm))]
describe(chain_vanilla_correlation[:, nms_vc, :])

In [ ]:
plot(chain_vanilla_correlation[:, nms_vc, :], size=(700, 600))

In [ ]:
# Prior predictive sampling
prior_chain_vc = Turing.sample(vanilla_correlation(nrow(d)), Prior(), 1_000)
prior_cors_vc = filter(!isnan, [cor(
    vec(Int.(Array(prior_chain_vc[:, "x[$i]", :]))),
    vec(Int.(Array(prior_chain_vc[:, "y[$i]", :])))
) for i in 1:nrow(d)])

# Posterior predictive sampling
post_cors_vc = filter(!isnan, [cor(
    vec(rand.(BernoulliLogit.(vec(Array(chain_vanilla_correlation[:, "z[1, $i]", :]))))),
    vec(rand.(BernoulliLogit.(vec(Array(chain_vanilla_correlation[:, "z[2, $i]", :])))))
) for i in 1:nrow(d)])

df_cor_vc = DataFrame(
    cor  = vcat(prior_cors_vc, post_cors_vc),
    type = vcat(fill("prior", length(prior_cors_vc)), fill("posterior", length(post_cors_vc)))
)
@df df_cor_vc density(:cor, group=:type,
    title="Vanilla correlation: prior vs posterior ρ",
    xlabel="Sample correlation", ylabel="Density")
vline!([cor(d.x, d.y)], label="observed", color=:red)

In [ ]:
# LOO log-likelihoods (y only, to compare with regression)
loglik_vanilla_correlation_y = hcat([
    logpdf.(BernoulliLogit.(chain_vanilla_correlation[:, "z[2, $i]", :]), d.y[i])
    for i in 1:nrow(d)
]...)

loo_vanilla_correlation = convert(
    Dict,
    R"library(loo); loo($(loglik_vanilla_correlation_y))"
)
println("LOOIC (vanilla correlation): ", loo_vanilla_correlation["looic"])

In [ ]:
model_vc = vanilla_correlation(nrow(d)) | (x=d.x, y=d.y)
ldf_vc = LogDensityFunction(model_vc)
samples_vc = Array(chain_vanilla_correlation[:, names(chain_vanilla_correlation, :parameters), :])
f_vc(x) = LogDensityProblems.logdensity(ldf_vc, x)
logml_vanilla_correlation = bridge_sampling(samples_vc, f_vc)
println("Log marginal likelihood (vanilla correlation): ", logml_vanilla_correlation)

The correlation model is preferred over the regression model in LOO. Neither controls for phylogenetic structure, however. We now turn to phylogenetically controlled approaches.

## Pagel & Meade style test for correlation (CTMC)

The CTMC-based test for correlated evolution (Pagel & Meade 2006) is the dominant method in phylogenetic linguistics. It models the joint evolution of $(x, y)$ as transitions among the four combined states $\{(0,0),(0,1),(1,0),(1,1)\}$ via a continuous-time Markov chain with rate matrix $Q$. Under the **independent** model the rates satisfy independence constraints (four free parameters); under the **dependent** model all eight off-diagonal rates are free. A Bayes factor between the two quantifies evidence for correlated evolution.

We implement Felsenstein's pruning algorithm in Julia directly.

The joint state encoding is: $\text{state} = 2x + y + 1 \in \{1,2,3,4\}$.

### Helper functions: rate matrix, stationary distribution, and Felsenstein pruning

The reference implementation separates three concerns: (1) building the $Q$ matrix from a vector of raw rates, (2) computing the stationary distribution via a linear solve, and (3) the pruning recursion itself. This factoring keeps each piece reusable and makes the Turing model definitions clean.

In [ ]:
const N_STATES_CTMC = 4
const N_RATES_INDEP = 4   # independent model
const N_RATES_DEP   = 8   # dependent model

"""
    rates_to_Q(rates, n_states)

Build a rate matrix Q from a vector of raw (positive) off-diagonal rates.
Rates are filled row-major upper triangle then lower triangle.
For the 4-state CTMC the layout matches the Pagel & Meade ordering:
  upper: (0,0)→(0,1), (0,0)→(1,0), (0,1)→(1,1), (1,0)→(1,1)
  lower: (0,1)→(0,0), (1,0)→(0,0), (1,1)→(0,1), (1,1)→(1,0)
The diagonal is set so that each row sums to zero.
"""
function rates_to_Q(rates::AbstractVector{T}, n::Int) where T <: Real
    Q = zeros(T, n, n)
    k = 1
    for i in 1:n, j in (i+1):n
        Q[i, j] = rates[k]; k += 1
    end
    for i in 2:n, j in 1:(i-1)
        Q[i, j] = rates[k]; k += 1
    end
    for i in 1:n
        Q[i, i] = -sum(Q[i, :])
    end
    return Q
end

"""
    stationary_dist(Q)

Compute the stationary distribution of Q by solving the over-determined
linear system [Q'; 1ᵀ] π = [0; 1] via least squares.
This is more numerically stable than a long-run matrix exponential.
"""
function stationary_dist(Q::AbstractMatrix{T}) where T <: Real
    n = size(Q, 1)
    A = vcat(Q', ones(T, 1, n))
    b = vcat(zeros(T, n), one(T))
    return A \ b
end

"""
    pruning_loglik(s, edges, edge_lengths, root_node, n_nodes, n_tips, Q, stat)

Felsenstein pruning log-likelihood for a 4-state CTMC.
- `s[i]`           : 1-based observed state at tip i (tips are nodes 1..n_tips)
- `edges`          : (Nedges × 2) matrix, 1-based local node indices, postorder
- `edge_lengths`   : branch lengths, same ordering as edges
- `root_node`      : index of the root node
- `n_nodes`        : total number of nodes
- `n_tips`         : number of tip taxa
- `Q`              : 4×4 rate matrix
- `stat`           : stationary distribution (length-4 vector)

Uses `ExponentialUtilities.exponential!` for the matrix exponential, which is
compatible with ForwardDiff dual numbers and therefore works with NUTS.
"""
function pruning_loglik(
    s::Vector{Int},
    edges::Matrix{Int},
    edge_lengths::Vector{Float64},
    root_node::Int,
    n_nodes::Int,
    n_tips::Int,
    Q::AbstractMatrix{T},
    stat::AbstractVector{T}
) where T <: Real
    # ll[node, state]: log conditional likelihood at each node
    ll = Matrix{T}(undef, n_nodes, N_STATES_CTMC)

    # Tip initialisation: hard assignment to observed state
    for i in 1:n_tips
        st = s[i]  # 1-based
        for k in 1:N_STATES_CTMC
            ll[i, k] = k == st ? zero(T) : T(-Inf)
        end
    end

    # Internal node initialisation: uninformative (log 1 = 0)
    for i in (n_tips+1):n_nodes, k in 1:N_STATES_CTMC
        ll[i, k] = zero(T)
    end

    # Post-order pruning
    for e in 1:size(edges, 1)
        parent = edges[e, 1]
        child  = edges[e, 2]

        # Transition probability matrix P(t) = expm(Q * t)
        # exponential! is in-place and ForwardDiff-compatible
        Pt = Q * edge_lengths[e]
        exponential!(Pt)

        for k in 1:N_STATES_CTMC
            acc = T(-Inf)
            for j in 1:N_STATES_CTMC
                v = log(max(Pt[k, j], eps(Float64))) + ll[child, j]
                # Skip impossible child states to avoid NaN in ForwardDiff derivatives
                isfinite(v) || continue
                acc = isinf(acc) ? v : logaddexp(acc, v)
            end
            ll[parent, k] += acc
        end
    end

    # Root marginal: weight by stationary distribution
    root_ll = T(-Inf)
    for k in 1:N_STATES_CTMC
        v = ll[root_node, k] + log(max(stat[k], eps(Float64)))
        isfinite(v) || continue
        root_ll = isinf(root_ll) ? v : logaddexp(root_ll, v)
    end
    return root_ll
end

### Encode tree for CTMC models

In [ ]:
# Build postorder edge list from the Phylo tree.
# Tips are assigned node indices 1..n_tips (in the order of `taxa`),
# internal nodes get indices n_tips+1..n_nodes, and the root index is returned.
function tree_to_edge_arrays(tree::RootedTree, taxa::Vector{String})
    root_name = getroot(tree).name
    n_tips    = length(taxa)

    # Assign tip indices 1..n_tips first, then internal nodes
    node_idx = Dict{String,Int}()
    for (i, t) in enumerate(taxa)
        node_idx[t] = i
    end
    next_idx = n_tips + 1
    for nd in traversal(tree, preorder)
        if !haskey(node_idx, nd.name)
            node_idx[nd.name] = next_idx
            next_idx += 1
        end
    end

    root_node = node_idx[root_name]
    n_nodes   = length(node_idx)

    parents  = Int[]
    children = Int[]
    lengths  = Float64[]

    for nd in traversal(tree, postorder)
        if !isroot(tree, nd)
            mother = getparent(tree, nd)
            br     = getinbound(tree, nd)
            bl     = getlength(tree, br)
            push!(parents,  node_idx[mother.name])
            push!(children, node_idx[nd.name])
            push!(lengths,  bl)
        end
    end

    edges = [parents children]

    (edges=edges, lengths=lengths,
     root_node=root_node, n_nodes=n_nodes, n_tips=n_tips,
     node_idx=node_idx)
end

ctmc_tree = tree_to_edge_arrays(edge_tree, taxa)
println("Root node index: ", ctmc_tree.root_node)
println("Number of edges: ", size(ctmc_tree.edges, 1))
println("Number of tips:  ", ctmc_tree.n_tips)

In [ ]:
# Encode observed tip states as joint (x,y) -> 1..4
# State encoding: state = 2x + y + 1
#   (x=0,y=0) -> 1,  (x=0,y=1) -> 2,  (x=1,y=0) -> 3,  (x=1,y=1) -> 4
tip_states_obs = 2 .* d.x .+ d.y .+ 1  # 1-based
println("Observed state distribution: ", countmap(tip_states_obs))

### Independent CTMC model

Under independence, the joint process on $(x, y)$ factorizes: the rate of transitioning in $x$ does not depend on the current value of $y$, and vice versa. This imposes four equality constraints on the 8 off-diagonal rates, leaving 4 free parameters. Using the `rates_to_Q` helper, the independent model is parameterized by 4 log-rates with a Normal(0,1) prior — equivalently a LogNormal(0,1) prior on the raw rates.

In [ ]:
@model function ctmc_independent(tip_states, ctmc_tree)
    # Normal(0,1) prior on log-rates ↔ LogNormal(0,1) on rates
    # 4 rates for the independent model: q01, q02, q13, q23 (upper triangle)
    # plus their reverses q10, q20, q31, q32 (lower triangle)
    # Independence constraint: rates share values across x-marginal and y-marginal.
    # We parameterize via 4 free log-rates and enforce the constraint when building Q.
    log_r ~ MvNormal(zeros(4), I)

    # Clamp to [-8, 8] (rates ≈ 0.0003–3000/myr) for numerical stability in expm
    lr = clamp.(log_r, -8.0, 8.0)

    # Independent model: 8 off-diagonal rates, but only 4 are free.
    # Rate layout (row-major upper then lower for 4×4):
    #   upper: q12=lr[1], q13=lr[2], q24=lr[3], q34=lr[4]
    #   lower: q21=lr[?], q31=lr[?], q42=lr[?], q43=lr[?]
    # Independence: q12=q34, q21=q43, q13=q24, q31=q42
    # (y-transition rates independent of x; x-transition rates independent of y)
    q_y0 = exp(lr[1])   # (x,0)->(x,1): same for x=0 and x=1
    q_y1 = exp(lr[2])   # (x,1)->(x,0): same for x=0 and x=1
    q_x0 = exp(lr[3])   # (0,y)->(1,y): same for y=0 and y=1
    q_x1 = exp(lr[4])   # (1,y)->(0,y): same for y=0 and y=1

    Q = zeros(eltype(lr), 4, 4)
    # States: 1=(0,0), 2=(0,1), 3=(1,0), 4=(1,1)
    Q[1, 2] = q_y0   # (0,0)->(0,1)
    Q[2, 1] = q_y1   # (0,1)->(0,0)
    Q[3, 4] = q_y0   # (1,0)->(1,1)  [same as (0,0)->(0,1)]
    Q[4, 3] = q_y1   # (1,1)->(1,0)  [same as (0,1)->(0,0)]
    Q[1, 3] = q_x0   # (0,0)->(1,0)
    Q[3, 1] = q_x1   # (1,0)->(0,0)
    Q[2, 4] = q_x0   # (0,1)->(1,1)  [same as (0,0)->(1,0)]
    Q[4, 2] = q_x1   # (1,1)->(0,1)  [same as (1,0)->(0,0)]
    for i in 1:4
        Q[i, i] = -sum(Q[i, :])
    end

    stat = try
        stationary_dist(Q)
    catch
        Turing.@addlogprob! -Inf
        return
    end

    ll = try
        pruning_loglik(
            tip_states,
            ctmc_tree.edges,
            ctmc_tree.lengths,
            ctmc_tree.root_node,
            ctmc_tree.n_nodes,
            ctmc_tree.n_tips,
            Q,
            stat
        )
    catch
        -Inf
    end
    Turing.@addlogprob! ll
end

In [ ]:
Random.seed!(42)
chain_ctmc_indep = Turing.sample(
    ctmc_independent(tip_states_obs, ctmc_tree),
    NUTS(0.65),
    MCMCThreads(),
    1_000,
    4
)
describe(chain_ctmc_indep)

In [ ]:
# Bridge sampling for marginal likelihood
model_ctmc_indep = ctmc_independent(tip_states_obs, ctmc_tree)
ldf_ctmc_indep = LogDensityFunction(model_ctmc_indep)
param_names_ci = names(chain_ctmc_indep, :parameters)
samples_ci = Array(chain_ctmc_indep[:, param_names_ci, :])
f_ci(x) = LogDensityProblems.logdensity(ldf_ctmc_indep, x)
logml_ctmc_indep = bridge_sampling(samples_ci, f_ci)
println("Log marginal likelihood (CTMC independent): ", logml_ctmc_indep)

### Dependent CTMC model

Under the dependent model, all 8 off-diagonal rates are free. We use the same `rates_to_Q` helper with 8 log-rates (Normal(0,1) prior), and the same pruning algorithm. The only difference is the prior is over 8 parameters rather than 4.

In [ ]:
@model function ctmc_dependent(tip_states, ctmc_tree)
    # 8 free log-rates, Normal(0,1) prior ↔ LogNormal(0,1) on rates
    log_r ~ MvNormal(zeros(8), I)

    # Clamp to [-8, 8] for numerical stability
    lr    = clamp.(log_r, -8.0, 8.0)
    rates = exp.(lr)

    # Pagel's dependent model: only single-feature transitions (no double flips).
    # States: 1=(0,0), 2=(0,1), 3=(1,0), 4=(1,1)
    # r[1..4] = forward rates; r[5..8] = reverse rates
    Q = zeros(eltype(rates), 4, 4)
    Q[1, 2] = rates[1]  # (0,0)→(0,1)
    Q[1, 3] = rates[2]  # (0,0)→(1,0)
    Q[2, 4] = rates[3]  # (0,1)→(1,1)
    Q[3, 4] = rates[4]  # (1,0)→(1,1)
    Q[2, 1] = rates[5]  # (0,1)→(0,0)
    Q[3, 1] = rates[6]  # (1,0)→(0,0)
    Q[4, 2] = rates[7]  # (1,1)→(0,1)
    Q[4, 3] = rates[8]  # (1,1)→(1,0)
    for i in 1:4
        Q[i, i] = -sum(Q[i, :])
    end
    stat = try
        stationary_dist(Q)
    catch
        Turing.@addlogprob! -Inf
        return
    end

    ll = try
        pruning_loglik(
            tip_states,
            ctmc_tree.edges,
            ctmc_tree.lengths,
            ctmc_tree.root_node,
            ctmc_tree.n_nodes,
            ctmc_tree.n_tips,
            Q,
            stat
        )
    catch
        -Inf
    end
    Turing.@addlogprob! ll
end

In [ ]:
Random.seed!(42)
chain_ctmc_dep = Turing.sample(
    ctmc_dependent(tip_states_obs, ctmc_tree),
    NUTS(0.65),
    MCMCThreads(),
    1_000,
    4
)
describe(chain_ctmc_dep)

In [ ]:
model_ctmc_dep = ctmc_dependent(tip_states_obs, ctmc_tree)
ldf_ctmc_dep = LogDensityFunction(model_ctmc_dep)
param_names_cd = names(chain_ctmc_dep, :parameters)
samples_cd = Array(chain_ctmc_dep[:, param_names_cd, :])
f_cd(x) = LogDensityProblems.logdensity(ldf_ctmc_dep, x)
logml_ctmc_dep = bridge_sampling(samples_cd, f_cd)
println("Log marginal likelihood (CTMC dependent): ", logml_ctmc_dep)

### CTMC Bayes factor

In [ ]:
bf_ctmc = logml_ctmc_indep - logml_ctmc_dep
println("Log Bayes factor (independent vs dependent): ", bf_ctmc)
if bf_ctmc > 0
    println("Evidence favors the INDEPENDENT model (no correlated evolution).")
else
    println("Evidence favors the DEPENDENT model (correlated evolution).")
end

The CTMC approach has one fundamental limitation: it treats typological features as discrete characters with a small number of states. This ignores the gradient nature of typological tendencies. We now turn to continuous models.

## Logistic regression with a phylogenetic random effect

### The Ornstein-Uhlenbeck process

The OU process is a stochastic process, similar to Brownian motion, but with a restoring force toward a cross-linguistic attractor. Its transition distribution is:

$$
X_t \sim \mathcal N\!\left(x_0 e^{-\lambda t} + \mu(1 - e^{-\lambda t}),\; \frac{\sigma}{\sqrt{2\lambda}}\sqrt{1 - e^{-2\lambda t}}\right)
$$

Parameters:
- $\mu$: long-run mean (cross-linguistic attractor)
- $\lambda$: rate of mean-reversion (strength of attractor)
- $\sigma$: diffusion coefficient

As $\lambda \to 0$, the OU process reduces to Brownian motion.

We add a phylogenetic random effect to the logistic regression:

$$
\begin{align}
\alpha &\sim \text{Student-t}(3, 0, 2.5)\\
\beta &\sim \mathcal N(0, 2)\\
\epsilon &\sim \text{OU}(\mu, \sigma, \lambda \mid \text{tree})\\
y_i &\sim \text{Bernoulli}(\text{logit}^{-1}(\alpha + \beta x_i + \epsilon_i))
\end{align}
$$

Rather than inverting a covariance matrix, we simulate the OU process node by node along the tree, starting from the root and propagating downward. This is equivalent to the covariance parameterization but is more efficient and easier to extend.

In [ ]:
@model function OU_regression(d, tree_info, taxa)
    N = length(taxa)
    root = tree_info.root
    mothers = tree_info.mothers
    daughters = tree_info.daughters
    lengths = tree_info.lengths
    nodes_dict = tree_info.nodes_dict
    n_nodes = length(nodes_dict)

    alpha ~ LocationScale(0.0, 2.5, TDist(3))
    beta  ~ Normal(0, 2)
    mu    ~ Normal(0, 2)
    sigma_u ~ Normal(0, 1)
    sigma := exp(sigma_u)
    lambda_u ~ Normal(0, 1)
    lambda := exp(lambda_u)

    # Standard normal reparameterization for all nodes
    z_std ~ filldist(Normal(), n_nodes)

    # OU process: root
    z = zeros(typeof(sigma), n_nodes)
    z[nodes_dict[root]] = mu + (sigma / sqrt(2 * lambda)) * z_std[nodes_dict[root]]

    # Propagate from root to tips (reverse postorder = preorder)
    for i in length(mothers):-1:1
        dgt = nodes_dict[daughters[i]]
        mth = nodes_dict[mothers[i]]
        len = lengths[i]
        decay = exp(-lambda * len)
        s = sigma * sqrt(-expm1(-2 * lambda * len) / (2 * lambda))
        mn = mu + (z[mth] - mu) * decay
        z[dgt] = mn + s * z_std[dgt]
    end

    # Likelihood at tips
    x_c = d.x .- mean(d.x)
    for (k, t) in enumerate(taxa)
        eta = alpha + beta * x_c[k] + z[nodes_dict[t]]
        d.y[k] ~ BernoulliLogit(eta)
    end
end

In [ ]:
Random.seed!(42)
chain_OU_regression = Turing.sample(
    OU_regression(d, tree_info, taxa),
    NUTS(0.95),
    MCMCThreads(),
    1_000,
    4
)

nms_our = [nm for nm in names(chain_OU_regression)
           if !occursin("z_std", string(nm)) && !occursin("_u", string(nm))]
describe(chain_OU_regression[:, nms_our, :])

In [ ]:
plot(chain_OU_regression[:, [:alpha, :beta, :mu, :sigma, :lambda], :], size=(700, 700))

In [ ]:
# LOO log-likelihoods
x_c = d.x .- mean(d.x)
loglik_OU_regression = hcat([begin
    eta = Array(chain_OU_regression[:, :alpha, :]) .+
          Array(chain_OU_regression[:, :beta, :]) .* x_c[i]
    # Note: z values are deterministic given z_std, but we use the posterior samples directly
    logpdf.(BernoulliLogit.(eta), d.y[i])
end for i in 1:nrow(d)]...)

loo_OU_regression = convert(
    Dict,
    R"library(loo); loo($(loglik_OU_regression))"
)
println("LOOIC (OU regression): ", loo_OU_regression["looic"])

In [ ]:
# Bridge sampling
model_our = OU_regression(d, tree_info, taxa)
ldf_our = LogDensityFunction(model_our)
samples_our = Array(chain_OU_regression[:, names(chain_OU_regression, :parameters), :])
f_our(x) = LogDensityProblems.logdensity(ldf_our, x)
logml_OU_regression = bridge_sampling(samples_our, f_our)
println("Log marginal likelihood (OU regression): ", logml_OU_regression)
println("Log BF (OU regression vs vanilla regression): ",
    logml_OU_regression - logml_vanilla_regression)

This is strong evidence in favor of the OU model over the vanilla regression.

### Brownian motion regression (baseline)

Brownian motion is the $\lambda \to 0$ limit of OU. We implement it using the precomputed Cholesky factor of the phylogenetic covariance matrix (VCV matrix).

$$
\begin{align}
\epsilon &\sim \mathcal N(0, \sigma^2 \cdot \text{VCV})\\
y_i &\sim \text{Bernoulli}(\text{logit}^{-1}(\alpha + \beta x_i + \epsilon_i))
\end{align}
$$

In [ ]:
@model function brownian_regression(d, scaled_chol, taxa)
    N = length(taxa)
    alpha ~ LocationScale(0.0, 2.5, TDist(3))
    beta  ~ Normal(0, 2)
    rate_u ~ Normal(0, 1)
    rate := exp(rate_u)

    z_std ~ filldist(Normal(), N)
    z := rate .* (scaled_chol * z_std)

    x_c = d.x .- mean(d.x)
    for i in 1:N
        d.y[i] ~ BernoulliLogit(alpha + beta * x_c[i] + z[i])
    end
end

In [ ]:
Random.seed!(42)
chain_brownian_regression = Turing.sample(
    brownian_regression(d, scaled_chol, taxa),
    NUTS(0.95),
    MCMCThreads(),
    1_000,
    4
)

nms_br = [nm for nm in names(chain_brownian_regression)
          if !occursin("z", string(nm)) && !occursin("_u", string(nm))]
describe(chain_brownian_regression[:, nms_br, :])

In [ ]:
# LOO and bridge sampling for brownian regression
loglik_brownian_regression = hcat([begin
    z_vals = Array(chain_brownian_regression[:, "z[$i]", :])
    x_c = d.x .- mean(d.x)
    logpdf.(BernoulliLogit.(
        Array(chain_brownian_regression[:, :alpha, :]) .+
        Array(chain_brownian_regression[:, :beta, :]) .* x_c[i] .+
        z_vals
    ), d.y[i])
end for i in 1:nrow(d)]...)

loo_brownian_regression = convert(
    Dict,
    R"library(loo); loo($(loglik_brownian_regression))"
)
println("LOOIC (Brownian regression): ", loo_brownian_regression["looic"])

In [ ]:
model_br = brownian_regression(d, scaled_chol, taxa)
ldf_br = LogDensityFunction(model_br)
samples_br = Array(chain_brownian_regression[:, names(chain_brownian_regression, :parameters), :])
f_br(x) = LogDensityProblems.logdensity(ldf_br, x)
logml_brownian_regression = bridge_sampling(samples_br, f_br)
println("Log marginal likelihood (Brownian regression): ", logml_brownian_regression)
println("Log BF (OU vs Brownian regression): ",
    logml_OU_regression - logml_brownian_regression)

The OU model is favored over Brownian motion — mean-reversion provides a genuinely better fit.

## Bivariate OU correlation model

The regression models above treat one variable as a predictor and the other as a response — an asymmetry hard to justify in observational typological data. A conceptually natural model treats both features symmetrically as co-evolving. If a diachronic dependency exists, it will manifest as a correlation between their evolutionary trajectories.

The bivariate OU correlation model uses the following covariance structure for the joint vector of latent variables across the two traits and $N$ languages:

$$
\Sigma_{(l_1, k_1),(l_2, k_2)} = R_{k_1, k_2} \cdot \frac{\sigma_{k_1}\sigma_{k_2}}{\lambda_{k_1}+\lambda_{k_2}}\left(1-e^{-(\lambda_{k_1}+\lambda_{k_2})t_{l_1,l_2}}\right)
$$

where $t_{l_1,l_2}$ is the patristic distance between languages $l_1$ and $l_2$, and $R$ is a $2\times 2$ correlation matrix.

We implement this by simulating the bivariate OU process along the tree, using correlated innovations at each branch:

In [ ]:
@model function OU_correlation_single_tree(tree_info, taxa, x, y)
    root       = tree_info.root
    mothers    = tree_info.mothers
    daughters  = tree_info.daughters
    lengths    = tree_info.lengths
    nodes_dict = tree_info.nodes_dict
    n_edges    = length(mothers)
    N          = length(taxa)

    # Correlation parameter
    rho_u ~ Normal()
    rho := 2 * cdf(Normal(), rho_u) - 1
    Sigma_l = [1.0 0.0; rho sqrt(1 - rho^2)]

    # Per-trait OU parameters
    sigma_u ~ filldist(Normal(), 2)
    sigma := exp.(sigma_u)
    lambda_u ~ filldist(Normal(), 2)
    lambda := exp.(lambda_u)

    # Trait means
    mu ~ MvNormal(zeros(2), 2.0)

    # Correlated innovations (one 2-vector per edge in the tree)
    z_std ~ filldist(MvNormal(zeros(n_edges), ones(n_edges)), 2)
    z_cor = z_std * Sigma_l   # n_edges × 2

    # Root states drawn from equilibrium distribution
    root_states ~ MvNormal(mu, sigma ./ sqrt.(2 .* lambda))

    # Propagate OU along tree
    n_nodes = length(nodes_dict)
    z = zeros(typeof(sigma[1]), n_nodes, 2)
    z[nodes_dict[root], :] .= root_states

    for i in length(mothers):-1:1
        dgt = nodes_dict[daughters[i]]
        mth = nodes_dict[mothers[i]]
        len = lengths[i]
        decay = exp.(-lambda .* len)
        s     = sigma .* sqrt.(-expm1.(-2 .* lambda .* len) ./ (2 .* lambda))
        mn    = mu .+ (z[mth, :] .- mu) .* decay
        z[dgt, :] = mn .+ s .* z_cor[i, :]
    end

    # Observed tip values
    x_logits = [z[nodes_dict[t], 1] for t in taxa]
    y_logits = [z[nodes_dict[t], 2] for t in taxa]
    x ~ product_distribution(BernoulliLogit.(x_logits))
    y ~ product_distribution(BernoulliLogit.(y_logits))
end

In [ ]:
Random.seed!(42)
chain_OU_correlation = Turing.sample(
    OU_correlation_single_tree(tree_info, taxa, d.x, d.y),
    MH(),
    MCMCThreads(),
    100_000,
    4;
    num_warmup=50_000,
    thinning=10
)

nms_ouc = [nm for nm in names(chain_OU_correlation)
           if !occursin("z", string(nm)) && !occursin("_u", string(nm)) &&
              !occursin("root_states", string(nm))]
describe(chain_OU_correlation[:, nms_ouc, :])

In [ ]:
hpd(chain_OU_correlation[:, ["mu[1]", "mu[2]", "sigma[1]", "sigma[2]",
                               "lambda[1]", "lambda[2]", "rho"], :])

In [ ]:
ridgelineplot(chain_OU_correlation, [:rho],
    title="Posterior distribution of ρ (OU correlation)")

Since the HPD of $\rho$ includes 0, we conclude there is no credible evidence for a diachronic correlation between the two variables.

### Prior and posterior predictive checks

In [ ]:
# Prior predictive
prior_chain_ouc = Turing.sample(
    OU_correlation_single_tree(tree_info, taxa, d.x, d.y),
    Prior(),
    length(chain_OU_correlation)
)

x_prior_ouc = Int.(hcat([vec(Array(prior_chain_ouc[:, "x[$i]", :])) for i in 1:nrow(d)]...))
y_prior_ouc = Int.(hcat([vec(Array(prior_chain_ouc[:, "y[$i]", :])) for i in 1:nrow(d)]...))

# Build node name list in taxa order for z extraction
z_tip_node_names = ["z[$(tree_info.nodes_dict[t]), 1]" for t in taxa]

x_post_ouc = hcat([
    vec(rand.(BernoulliLogit.(vec(Array(chain_OU_correlation[:, "z[$(tree_info.nodes_dict[t]), 1]", :])))))
    for t in taxa
]...)

y_post_ouc = hcat([
    vec(rand.(BernoulliLogit.(vec(Array(chain_OU_correlation[:, "z[$(tree_info.nodes_dict[t]), 2]", :])))))
    for t in taxa
]...)

prior_cors_ouc = filter(!isnan, [cor(x_prior_ouc[i, :], y_prior_ouc[i, :]) for i in 1:size(x_prior_ouc, 1)])
post_cors_ouc  = filter(!isnan, [cor(x_post_ouc[i, :],  y_post_ouc[i, :])  for i in 1:size(x_post_ouc, 1)])

df_ouc = DataFrame(
    cor  = vcat(prior_cors_ouc, post_cors_ouc),
    type = vcat(fill("prior", length(prior_cors_ouc)), fill("posterior", length(post_cors_ouc)))
)
@df df_ouc density(:cor, group=:type,
    title="OU correlation: prior vs posterior ρ",
    xlabel="Sample correlation", ylabel="Density")
vline!([cor(d.x, d.y)], label="observed", color=:red)

### LOO and bridge sampling for OU correlation

In [ ]:
loglik_OU_correlation_y = hcat([
    logpdf.(BernoulliLogit.(
        chain_OU_correlation[:, "z[$(tree_info.nodes_dict[taxa[i]]), 2]", :]
    ), d.y[i])
    for i in 1:nrow(d)
]...)

loo_OU_correlation = convert(
    Dict,
    R"library(loo); loo($(loglik_OU_correlation_y))"
)
println("LOOIC (OU correlation): ", loo_OU_correlation["looic"])

In [ ]:
model_ouc = OU_correlation_single_tree(tree_info, taxa, d.x, d.y)
ldf_ouc = LogDensityFunction(model_ouc)

samples_ouc = Array(chain_OU_correlation[:, names(chain_OU_correlation, :parameters), :])
f_ouc(x) = LogDensityProblems.logdensity(ldf_ouc, x)
logml_OU_correlation = bridge_sampling(samples_ouc, f_ouc)
println("Log marginal likelihood (OU correlation): ", logml_OU_correlation)
println("Log BF (OU correlation vs vanilla correlation): ",
    logml_OU_correlation - logml_vanilla_correlation)

Again we find strong evidence in favor of the phylogenetic model.

## Brownian motion correlation model

For completeness, we also fit the Brownian motion correlation model using the VCV matrix.

In [ ]:
@model function brownian_correlation_single(N, scaled_chol)
    rho_u ~ Normal()
    rho := 2 * cdf(Normal(), rho_u) - 1
    Sigma_l = [1.0 0.0; rho sqrt(1 - rho^2)]
    rates_u ~ filldist(Normal(), 2)
    rates := exp.(rates_u)
    mu ~ MvNormal(zeros(2), 2.0)

    z_std ~ filldist(MvNormal(zeros(N), ones(N)), 2)
    z := diagm(rates) * Sigma_l * z_std * scaled_chol' .+ mu

    x ~ product_distribution(BernoulliLogit.(z[1, :]))
    y ~ product_distribution(BernoulliLogit.(z[2, :]))
end

In [ ]:
Random.seed!(42)
chain_brownian_correlation = Turing.sample(
    brownian_correlation_single(nrow(d), scaled_chol) | (x=d.x, y=d.y),
    MH(),
    MCMCThreads(),
    100_000,
    4;
    num_warmup=50_000,
    thinning=10
)

nms_bc = [nm for nm in names(chain_brownian_correlation)
          if !occursin("z", string(nm)) && !occursin("_u", string(nm))]
describe(chain_brownian_correlation[:, nms_bc, :])

In [ ]:
loglik_brownian_correlation_y = hcat([
    logpdf.(BernoulliLogit.(chain_brownian_correlation[:, "z[2, $i]", :]), d.y[i])
    for i in 1:nrow(d)
]...)

loo_brownian_correlation = convert(
    Dict,
    R"library(loo); loo($(loglik_brownian_correlation_y))"
)
println("LOOIC (Brownian correlation): ", loo_brownian_correlation["looic"])

In [ ]:
model_bc = brownian_correlation_single(nrow(d), scaled_chol) | (x=d.x, y=d.y)
ldf_bc = LogDensityFunction(model_bc)

samples_bc = Array(chain_brownian_correlation[:, names(chain_brownian_correlation, :parameters), :])
f_bc(x) = LogDensityProblems.logdensity(ldf_bc, x)
logml_brownian_correlation = bridge_sampling(samples_bc, f_bc)
println("Log marginal likelihood (Brownian correlation): ", logml_brownian_correlation)
println("Log BF (OU correlation vs Brownian correlation): ",
    logml_OU_correlation - logml_brownian_correlation)

## Final comparison

### Bayes factors

In [ ]:
println("=== Log marginal likelihoods ===")
println("Vanilla regression:      ", logml_vanilla_regression)
println("Vanilla correlation:     ", logml_vanilla_correlation)
println("CTMC independent:        ", logml_ctmc_indep)
println("CTMC dependent:          ", logml_ctmc_dep)
println("Brownian regression:     ", logml_brownian_regression)
println("Brownian correlation:    ", logml_brownian_correlation)
println("OU regression:           ", logml_OU_regression)
println("OU correlation:          ", logml_OU_correlation)

println()
println("=== Key Bayes factors (log scale) ===")
println("OU correlation vs CTMC independent:     ", logml_OU_correlation - logml_ctmc_indep)
println("OU correlation vs Brownian correlation: ", logml_OU_correlation - logml_brownian_correlation)
println("OU regression vs Brownian regression:   ", logml_OU_regression - logml_brownian_regression)
println("OU regression vs vanilla regression:    ", logml_OU_regression - logml_vanilla_regression)

### LOO comparison

In [ ]:
loo_results = DataFrame(
    model = [
        "vanilla_regression",
        "vanilla_correlation",
        "CTMC_independent",
        "CTMC_dependent",
        "brownian_regression",
        "brownian_correlation",
        "OU_regression",
        "OU_correlation"
    ],
    LOOIC = [
        loo_vanilla_regression["looic"],
        loo_vanilla_correlation["looic"],
        NaN,   # computed separately in R if needed
        NaN,
        loo_brownian_regression["looic"],
        loo_brownian_correlation["looic"],
        loo_OU_regression["looic"],
        loo_OU_correlation["looic"]
    ]
)

sort!(dropmissing(loo_results, :LOOIC), :LOOIC)

Pareto-smoothed leave-one-out cross-validation and Bayes factors measure different things:

- The **Bayes Factor** evaluates the joint model of both variables together. A model that fits the joint distribution better — capturing how $x$ and $y$ co-evolve — will be favored.
- The **LOOIC** measures predictive accuracy for $y$ alone, given $x$. The CTMC models, despite their coarser representation, can make relatively sharp predictions about individual left-out languages.

The results are consistent: continuous latent-variable models substantially outperform CTMC. Among the continuous models, OU outperforms Brownian motion.

**Practical recommendation:** For binary typological features, the OU correlation model is the preferred choice when the research question concerns co-evolution or correlation. The OU regression model is appropriate when the question is directional. In both cases the Turing.jl implementation is straightforward to adapt to other feature pairs, other phylogenies, or additional random effects (geographic, family-level, etc.).

## Final thoughts

All the models discussed here use a single phylogeny (the EDGE MCC tree). Averaging over phylogenetic uncertainty — using a posterior sample of trees — can be done in Turing.jl by sampling a tree index as a discrete latent variable, as shown in the companion notebook `phylogenetic_OU_regression_julia.qmd`. That notebook marginalizes over 902 posterior EDGE trees using Metropolis-Hastings with a continuous tree-index parameterization.

Another advantage of explicit Turing.jl implementations is extensibility: geographic random effects, additional hierarchical levels, or non-binary responses can all be added by extending the model function.

More details can be found in *Computational Typology* (https://arxiv.org/abs/2504.15642).

## References

Bouckaert, R., Redding, D., Sheehan, O., Kyritsis, T., Gray, R., Jones, K. E., & Atkinson, Q. (2022). Global language diversification is linked to socio-ecology and threat status. SocArXiv. https://osf.io/f8tr6/download

Dryer, M. S. (1992). The Greenbergian word order correlations. *Language*, 68(1), 81–138. https://doi.org/10.2307/416463

Pagel, M., & Meade, A. (2006). Bayesian analysis of correlated evolution of discrete characters by reversible-jump Markov chain Monte Carlo. *The American Naturalist*, 167(6), 808–825. https://doi.org/10.1086/503444